# 01. Data Preprocessing & Validation
This notebook handles the loading, cleaning, text sanitization, duplicate resolution, and creation of text preprocessing variants for the Democrat vs. Republican Twitter dataset.

### Step 1: Initial Dataset Loading
We load the raw congressional tweets from `ExtractedTweets.csv` and inspect class ratios.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path('/kaggle/input/democratvsrepublicantweets/ExtractedTweets.csv')
df_raw = pd.read_csv(csv_path)
print(f"Raw Dataset Shape: {df_raw.shape}")
print("\n--- Initial Party Label Counts ---")
print(df_raw['Party'].value_counts(dropna=False))

### Step 2: Data Cleaning and Label Normalization
We sanitize the dataset by dropping nulls, retaining only valid party classes, removing conflicting tweets, and dropping exact duplicates.

In [ ]:
initial_count = len(df_raw)
df_clean = df_raw.dropna(subset=['Tweet', 'Party']).copy()
print(f"Dropped {initial_count - len(df_clean)} row(s) with missing values.")

valid_labels = {'Democrat', 'Republican'}
df_clean = df_clean[df_clean['Party'].isin(valid_labels)]

tweet_groups = df_clean.groupby('Tweet')['Party'].nunique()
conflicting_tweets = tweet_groups[tweet_groups > 1].index
if len(conflicting_tweets) > 0:
    pre_conflict_count = len(df_clean)
    df_clean = df_clean[~df_clean['Tweet'].isin(conflicting_tweets)]
    print(f"Dropped {pre_conflict_count - len(df_clean)} row(s) due to conflicting labels.")

pre_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Tweet', 'Party', 'Handle'])
print(f"Dropped {pre_dedup - len(df_clean)} exact duplicate rows.")
print(f"Final Cleaned Dataset Size: {df_clean.shape}")

### Step 3: Text Preprocessing Variants
We create three text variants to analyze political text signatures under different punctuation constraints without using stemming or lemmatization.

In [ ]:
import re
import string

df_clean['tweet_original'] = df_clean['Tweet'].copy()

def clean_variant_a(text):
    return re.sub(r'\s+', ' ', str(text)).lower().strip()

def clean_variant_b(text):
    text = str(text).lower()
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    text = re.sub(r'[@#](\w+)', r'\1', text)
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    return re.sub(r'\s+', ' ', text).strip()

def clean_variant_c(text):
    text = str(text).lower()
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()

df_clean['tweet_variant_a'] = df_clean['tweet_original'].apply(clean_variant_a)
df_clean['tweet_variant_b'] = df_clean['tweet_original'].apply(clean_variant_b)
df_clean['tweet_variant_c'] = df_clean['tweet_original'].apply(clean_variant_c)

print("Preprocessing variants generated!")
display(df_clean[['tweet_original', 'tweet_variant_b']].head(2))

### Step 4: Stratified Splitting and Leakage Prevention
To guarantee zero cross-split duplicate text leakage, we drop duplicate text bodies before doing an 80/10/10 stratified split.

In [ ]:
from sklearn.model_selection import train_test_split

df_unique = df_clean.drop_duplicates(subset=['tweet_original']).copy()
print(f"Unique dataset size: {df_unique.shape[0]} rows")

train_df, temp_df = train_test_split(
    df_unique, test_size=0.2, random_state=42, stratify=df_unique['Party']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['Party']
)

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

### Step 5: Save Processed Outputs
We persist our cleaned, split files inside `/content/` for training.

In [ ]:
output_dir = Path('/content')
df_clean.to_csv(output_dir / 'cleaned_dataset_all_variants.csv', index=False)
train_df.to_csv(output_dir / 'train_split.csv', index=False)
val_df.to_csv(output_dir / 'val_split.csv', index=False)
test_df.to_csv(output_dir / 'test_split.csv', index=False)

print("All split files successfully saved to /content/!")